# Week 09 Demo: Siamese Networks & Semantic Search

Welcome to the demonstration for Week 9! This notebook walks through every concept from the lecture in runnable code, aligned with slide order.

**Sections (aligned with slide order):**
1. Setup & Initialization
2. The Anisotropy Problem — raw BERT fails *(Demo Slide 12)*
   - 2b. Measuring the cone properly — mean pairwise cosine over many pairs
3. Manual Mean Pooling with `attention_mask` *(Demo Slide 16)*
4. `sentence-transformers` Library — Industry Standard API *(Slide 20)*
5. Manual SBERT Pipeline — Full 5-Step Implementation + Verification *(Slide 21)*
6. Triplet Loss & Margin Ranking Loss — distance vs. similarity form *(Slides 27, 29)*
7. Building a Vector Index with FAISS *(Slide 41)*


## 1. Setup & Initialization
First, let's load a standard pre-trained BERT model. We will NOT use `sentence-transformers` yet, to see the raw output.

In [1]:
import os
os.environ['KMP_DUPLICATE_LIB_OK']='True'
import torch
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModel

# Load standard BERT (NOT fine-tuned for sentence embeddings!)
model_name = "bert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModel.from_pretrained(model_name)



/Users/nontkanungsukkasem/anaconda3/lib/python3.11/site-packages/pandas/core/computation/expressions.py:23: UserWarning: Pandas requires version '2.10.2' or newer of 'numexpr' (version '2.8.7' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED
/Users/nontkanungsukkasem/anaconda3/lib/python3.11/site-packages/pandas/core/arrays/masked.py:56: UserWarning: Pandas requires version '1.4.2' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


## 2. The Anisotropy Problem (Why raw BERT fails)
Let's take three sentences. Sentences A and C are related (animals). Sentence B is unrelated.
Let's see what raw BERT's `[CLS]` token thinks about their similarities.

Watch two things: the **ranking** (does the related pair win?) and the **floor** (does anything ever score low?).


In [2]:
sentences = [
    "The cat sat on the mat.",               # A
    "I love training deep neural networks.", # B (UNRELATED to A)
    "Dogs are chasing cats in the park."     # C (RELATED to A)
]

# Tokenize
inputs = tokenizer(sentences, padding=True, truncation=True, return_tensors="pt")

# Forward pass
with torch.no_grad():
    outputs = model(**inputs)

# Extract the [CLS] token embedding for each sentence
# outputs.last_hidden_state shape: [batch_size, seq_len, hidden_size]
cls_embeddings = outputs.last_hidden_state[:, 0, :] # Get the 0th token [CLS]

print("Shape of CLS embeddings:", cls_embeddings.shape)

# Calculate Cosine Similarities
sim_A_B = F.cosine_similarity(cls_embeddings[0].unsqueeze(0), cls_embeddings[1].unsqueeze(0))
sim_A_C = F.cosine_similarity(cls_embeddings[0].unsqueeze(0), cls_embeddings[2].unsqueeze(0))

print(f"Similarity (A vs B - UNRELATED): {sim_A_B.item():.4f}")
print(f"Similarity (A vs C - RELATED)  : {sim_A_C.item():.4f}")

print()
if sim_A_B.item() > sim_A_C.item():
    print("Look at the ORDER, not just the size of the numbers:")
    print("  raw BERT scores the UNRELATED pair HIGHER than the related pair.")
    print("  The ranking is INVERTED. Cosine on raw BERT is not merely noisy - it is wrong.")
else:
    print("The ranking happens to be correct for this pair - run 2b below, the ordering")
    print("of two hand-picked pairs is not a reliable signal either way.")

print()
print("Also note the FLOOR: nothing scores low. Two sentences with nothing in common")
print("still sit well above 0. Section 2b measures that floor over many pairs.")


Shape of CLS embeddings: torch.Size([3, 768])
Similarity (A vs B - UNRELATED): 0.8610
Similarity (A vs C - RELATED)  : 0.7942

Look at the ORDER, not just the size of the numbers:
  raw BERT scores the UNRELATED pair HIGHER than the related pair.
  The ranking is INVERTED. Cosine on raw BERT is not merely noisy - it is wrong.

Also note the FLOOR: nothing scores low. Two sentences with nothing in common
still sit well above 0. Section 2b measures that floor over many pairs.


### 2b. Measuring the cone properly *(the number that belongs on Slide 19)*

Two sentence pairs prove nothing. The claim on the slides is about the **whole distribution**:
raw BERT vectors sit in a narrow cone, so *every* pair looks similar.

That is a statement about the mean and — more importantly — the **minimum** pairwise cosine
over many pairs. Let's measure it, for three encoders, on the same 12 sentences.


In [3]:
import itertools

probe_sentences = [
    "The cat is sleeping on the couch.",
    "The stock market crashed today.",
    "A feline is resting on the sofa.",
    "I love training deep neural networks.",
    "Dogs chase cats in the park.",
    "The recipe for chocolate chip cookies.",
    "Global warming causes severe weather changes.",
    "Apple announces new iPhone features.",
    "How to bake the perfect dessert.",
    "Deep learning achieves state of the art in NLP.",
    "Stock prices soar after tech earnings.",
    "Climate change impacts marine life.",
]

def pairwise_stats(emb, label):
    """emb: [n, d] L2-normalized -> print the distribution of all n*(n-1)/2 pairwise cosines."""
    sim = emb @ emb.T
    iu  = torch.triu_indices(emb.shape[0], emb.shape[0], offset=1)
    v   = sim[iu[0], iu[1]]
    print(f"{label:34s} pairs={len(v):3d}  mean={v.mean():+.3f}  min={v.min():+.3f}  "
          f"max={v.max():+.3f}  range={(v.max()-v.min()):.3f}")
    return v

# --- Encoder 1 & 2: raw BERT, both pooling strategies ---
probe_inputs = tokenizer(probe_sentences, padding=True, truncation=True, return_tensors="pt")
with torch.no_grad():
    probe_hidden = model(**probe_inputs).last_hidden_state

raw_cls  = F.normalize(probe_hidden[:, 0, :], p=2, dim=1)
_m       = probe_inputs['attention_mask'].unsqueeze(-1).expand(probe_hidden.size()).float()
raw_mean = F.normalize((probe_hidden * _m).sum(1) / _m.sum(1).clamp(min=1e-9), p=2, dim=1)

# --- Encoder 3: a properly trained SBERT ---
from sentence_transformers import SentenceTransformer
_sb  = SentenceTransformer('all-MiniLM-L6-v2')
sbert_emb = torch.tensor(_sb.encode(probe_sentences, normalize_embeddings=True))

print("Pairwise cosine distribution over 12 sentences (66 pairs):\n")
v_cls  = pairwise_stats(raw_cls,   "raw BERT  [CLS]")
v_mean = pairwise_stats(raw_mean,  "raw BERT  mean-pooled")
v_sb   = pairwise_stats(sbert_emb, "SBERT     (all-MiniLM-L6-v2)")

print()
print("Read the MIN column, not the mean:")
print(f"  raw BERT never scores any pair below {v_mean.min():+.3f} - nothing is ever 'dissimilar'.")
print(f"  SBERT pushes unrelated pairs down to {v_sb.min():+.3f}.")
print(f"  Raw BERT squeezes 66 pairs into a band {(v_mean.max()-v_mean.min()):.2f} wide; "
      f"SBERT spreads them over {(v_sb.max()-v_sb.min()):.2f}.")
print()
print("THAT is the cone. Not 'every score is 0.99' - but 'no score is ever low,")
print("so cosine has no room left to separate anything'.")


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Pairwise cosine distribution over 12 sentences (66 pairs):

raw BERT  [CLS]                    pairs= 66  mean=+0.768  min=+0.561  max=+0.942  range=0.380
raw BERT  mean-pooled              pairs= 66  mean=+0.580  min=+0.457  max=+0.848  range=0.391
SBERT     (all-MiniLM-L6-v2)       pairs= 66  mean=+0.057  min=-0.156  max=+0.556  range=0.712

Read the MIN column, not the mean:
  raw BERT never scores any pair below +0.457 - nothing is ever 'dissimilar'.
  SBERT pushes unrelated pairs down to -0.156.
  Raw BERT squeezes 66 pairs into a band 0.39 wide; SBERT spreads them over 0.71.

THAT is the cone. Not 'every score is 0.99' - but 'no score is ever low,
so cosine has no room left to separate anything'.


## 3. Manual Mean Pooling (Anti-Black-Box)
Instead of `[CLS]`, SBERT uses **Mean Pooling** over all token embeddings. 
Crucially, we must ignore the padding tokens (`[PAD]`) using the `attention_mask`.

In [4]:
def mean_pooling(token_embeddings, attention_mask):
    """
    token_embeddings: [batch_size, seq_len, hidden_size]
    attention_mask:   [batch_size, seq_len] (1 for real tokens, 0 for padding)
    """
    # Step 1: Expand attention mask to match the hidden_size dimension
    # [batch_size, seq_len] -> [batch_size, seq_len, 1] -> [batch_size, seq_len, hidden_size]
    input_mask_expanded = attention_mask.unsqueeze(-1).expand(token_embeddings.size()).float()

    # Step 2: Multiply embeddings by the mask to zero out padding tokens
    masked_embeddings = token_embeddings * input_mask_expanded

    # Step 3: Sum the embeddings along the sequence length (dim=1)
    # [batch_size, hidden_size]
    sum_embeddings = torch.sum(masked_embeddings, dim=1)

    # Step 4: Sum the mask to know how many actual tokens we have per sentence
    # clamp to avoid division by zero
    sum_mask = torch.clamp(input_mask_expanded.sum(dim=1), min=1e-9)

    # Step 5: Calculate the mean
    mean_pooled = sum_embeddings / sum_mask

    return mean_pooled

token_embeddings = outputs.last_hidden_state
attention_mask = inputs['attention_mask']
# print inputs token to compare with attention mask
print("Input tokens:", tokenizer.convert_ids_to_tokens(inputs['input_ids'][0]))
print("Attention mask:", attention_mask[0])

mean_embeddings = mean_pooling(token_embeddings, attention_mask)
print("Shape of Mean-Pooled embeddings:", mean_embeddings.shape)

# --- WHY the mask matters: [PAD] outputs are NOT near-zero vectors ---
# A common misconception: "[PAD] embeddings are ~0, so averaging them just shrinks the vector."
# Not true. Padded positions are masked as attention KEYS, but they still carry position
# embeddings, still pass through every feed-forward and residual layer, and still attend
# TO the real tokens. They come out large and meaningless. Let's measure it.
_pad_rows  = token_embeddings[attention_mask == 0]
_real_rows = token_embeddings[attention_mask == 1]
print()
print("Why we must mask -- the size of the vectors we are throwing away:")
print(f"  mean L2 norm of REAL token outputs : {_real_rows.norm(dim=-1).mean():.2f}")
print(f"  mean L2 norm of [PAD] token outputs: {_pad_rows.norm(dim=-1).mean():.2f}")
print("  -> [PAD] outputs are NOT near zero. Averaging them in does not shrink the")
print("     sentence vector toward 0 - it CORRUPTS it with large, meaningless directions.")

# Note: Even with correct mean pooling, if the model isn't fine-tuned with contrastive loss,
# the vectors will still be anisotropic (see Section 2b).


Input tokens: ['[CLS]', 'the', 'cat', 'sat', 'on', 'the', 'mat', '.', '[SEP]', '[PAD]']
Attention mask: tensor([1, 1, 1, 1, 1, 1, 1, 1, 1, 0])
Shape of Mean-Pooled embeddings: torch.Size([3, 768])

Why we must mask -- the size of the vectors we are throwing away:
  mean L2 norm of REAL token outputs : 14.90
  mean L2 norm of [PAD] token outputs: 9.67
  -> [PAD] outputs are NOT near zero. Averaging them in does not shrink the
     sentence vector toward 0 - it CORRUPTS it with large, meaningless directions.


### Verification: No Hidden "Magic" (Slide 16)

In [5]:
from sentence_transformers import SentenceTransformer

# Load the exact same raw BERT model through the official library
sbert_model = SentenceTransformer('bert-base-uncased')
sbert_embeddings = sbert_model.encode(sentences, convert_to_tensor=True)

# Move to CPU for comparison
mean_cpu  = mean_embeddings.cpu()
sbert_cpu = sbert_embeddings.cpu()

print("=== Verification: Manual Mean Pooling vs. Library ===")
print("Manual pooling (First 5 dims):", mean_cpu[0][:5].tolist())
print("Library output (First 5 dims):", sbert_cpu[0][:5].tolist())

# allclose, not equality: the two paths do the same math in a different order,
# so they agree to ~1e-7, not bit-for-bit.
is_identical = torch.allclose(mean_cpu, sbert_cpu, atol=1e-4)
if is_identical:
    print("\n✅ YES! Manual mean pooling matches the library to floating-point precision!")
    print("   The library has no magic — it's just automating the math we built above.")
else:
    print("\n❌ NO! They don't match.")


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


=== Verification: Manual Mean Pooling vs. Library ===
Manual pooling (First 5 dims): [-0.1818033903837204, -0.26617854833602905, -0.2188664823770523, 0.2108875960111618, 0.28473350405693054]
Library output (First 5 dims): [-0.18180376291275024, -0.2661784589290619, -0.21886594593524933, 0.21088695526123047, 0.2847340703010559]

✅ YES! Manual mean pooling matches the library to floating-point precision!
   The library has no magic — it's just automating the math we built above.


## 4. `sentence-transformers` Library — Industry Standard API (Slide 20)

After understanding how mean pooling works manually, see how the `sentence-transformers` library makes it **2 lines**:

```python
model = SentenceTransformer('model-name')
embeddings = model.encode(sentences, normalize_embeddings=True)  # shape: [n, dim]
```

The library runs all 5 SBERT steps automatically. The cell below compares sentence pairs across different production models from the slide's model table.

In [6]:
from sentence_transformers import SentenceTransformer
import numpy as np

test_sentences = [
    "A man is eating food.",
    "A man is eating pasta.",         # semantically close to #1
    "The girl is carrying a baby.",   # semantically different
]

# Production model — fast, 384-dim, trained on 1B+ sentence pairs
model_mini = SentenceTransformer('all-MiniLM-L6-v2')
emb_mini   = model_mini.encode(test_sentences, normalize_embeddings=True)

sim_12 = np.dot(emb_mini[0], emb_mini[1])  # eating food vs. eating pasta
sim_13 = np.dot(emb_mini[0], emb_mini[2])  # eating food vs. carrying baby

print(f"Model: all-MiniLM-L6-v2  (dim={emb_mini.shape[1]})")
print(f"  'eating food'  vs  'eating pasta' : {sim_12:.4f}  ← HIGH  ✓")
print(f"  'eating food'  vs  'carrying baby': {sim_13:.4f}  ← LOW   ✓")
print(f"  Output shape: {emb_mini.shape}")
print()
print("--- Same API, different models (just swap the model name string) ---")
print("  SentenceTransformer('all-MiniLM-L6-v2')       → dim=384, fast, general")
print("  SentenceTransformer('all-mpnet-base-v2')      → dim=768, balanced")
print("  SentenceTransformer('BAAI/bge-large-en-v1.5') → dim=1024, high quality")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Model: all-MiniLM-L6-v2  (dim=384)
  'eating food'  vs  'eating pasta' : 0.7035  ← HIGH  ✓
  'eating food'  vs  'carrying baby': -0.1050  ← LOW   ✓
  Output shape: (3, 384)

--- Same API, different models (just swap the model name string) ---
  SentenceTransformer('all-MiniLM-L6-v2')       → dim=384, fast, general
  SentenceTransformer('all-mpnet-base-v2')      → dim=768, balanced
  SentenceTransformer('BAAI/bge-large-en-v1.5') → dim=1024, high quality


## 5. The Manual SBERT *Pipeline* — Full 5-Step Implementation (Slide 21)

Now let's build the complete SBERT pipeline from scratch, covering all **5 steps** from Slide 21:

| Step | Code | Output shape |
|---|---|---|
| 1. Load BERT | `AutoModel.from_pretrained(...)` | model |
| 2. Tokenize | `tokenizer(sentences, padding=True)` | `input_ids`, `attention_mask` |
| 3. Forward Pass | `model(**inputs).last_hidden_state` | `[B, T, 768]` |
| 4. Masked Mean Pool | zero out `[PAD]`, average real tokens | `[B, 768]` |
| 5. L2 Normalize | `F.normalize(embedding, p=2, dim=1)` | `[B, 768]`, all norms = 1 |

> ⚠️ **These 5 steps are SBERT's *architecture*, not SBERT itself.** We are still running
> `bert-base-uncased` here. Pooling and normalizing raw BERT does **not** fix anisotropy —
> Section 2b proved that. What turns this pipeline into SBERT is **contrastive fine-tuning**
> (Slides 22–33). The steps below are the inference-time plumbing; the training is what
> reshapes the geometry.

Steps 1–4 were done in Section 3. Below we add **Step 5** and verify the full pipeline matches the `sentence-transformers` library.


In [7]:
import torch.nn.functional as F
# --- Step 5: L2 Normalize (Slide 21) ---
# SBERT adds one final step: L2 Normalization, so that dot product equals cosine similarity.
normalized_embeddings = F.normalize(mean_embeddings, p=2, dim=1)

print("=== 1. See how the raw values change ===")
# show the first 5 out of 768 dimensions of the first sentence embedding before and after normalization
print(f"Raw values BEFORE: {mean_embeddings[0][:5].tolist()}")
print(f"Raw values AFTER:  {normalized_embeddings[0][:5].tolist()}")

print("\n=== 2. Check the Norm (Magnitude of the vector) ===")
print("The .norm() function calculates the Magnitude (distance from origin).")
print("Notice how the Magnitude becomes 1.0 (subject to minor FP32 floating-point rounding errors).")
print(f"Magnitude BEFORE: {mean_embeddings.norm(dim=1).tolist()}")
print(f"Magnitude AFTER:  {normalized_embeddings.norm(dim=1).tolist()}  ← all ≈ 1.0 ✓")

print("\n=== 3. Verify with the Library ===")
# We ask the library to also normalize its output for a fair comparison
sbert_normalized = sbert_model.encode(sentences, convert_to_tensor=True, normalize_embeddings=True)
print(f"Manual normalized: {normalized_embeddings[0][:5].tolist()}")
print(f"Library output:    {sbert_normalized.cpu()[0][:5].tolist()}")


=== 1. See how the raw values change ===
Raw values BEFORE: [-0.1818033903837204, -0.26617854833602905, -0.2188664823770523, 0.2108875960111618, 0.28473350405693054]
Raw values AFTER:  [-0.017806772142648697, -0.026070915162563324, -0.02143692411482334, 0.020655430853366852, 0.027888284996151924]

=== 2. Check the Norm (Magnitude of the vector) ===
The .norm() function calculates the Magnitude (distance from origin).
Notice how the Magnitude becomes 1.0 (subject to minor FP32 floating-point rounding errors).
Magnitude BEFORE: [10.209789276123047, 8.627603530883789, 10.503263473510742]
Magnitude AFTER:  [0.9999999403953552, 0.9999999403953552, 1.0000001192092896]  ← all ≈ 1.0 ✓

=== 3. Verify with the Library ===
Manual normalized: [-0.017806772142648697, -0.026070915162563324, -0.02143692411482334, 0.020655430853366852, 0.027888284996151924]
Library output:    [-0.01780681312084198, -0.026070913299918175, -0.02143687754869461, 0.020655373111367226, 0.02788834646344185]


## 6. Triplet Loss & Margin Ranking Loss — the same idea in two coordinate systems
*(Slides 27 & 29)*

Margin Ranking Loss is the **similarity-domain** twin of Triplet Loss, which is written in **distances**.

| | Formula | Wants |
|---|---|---|
| Triplet Loss | $\max(0,\; d(a,p) - d(a,n) + \alpha)$ | $d(a,p)$ small, $d(a,n)$ large |
| Margin Ranking Loss | $\max(0,\; \text{sim}(a,n) - \text{sim}(a,p) + \alpha)$ | $\text{sim}(a,p)$ high, $\text{sim}(a,n)$ low |

With **cosine distance** ($d = 1 - \text{sim}$) the two are algebraically identical: substituting
$d = 1 - \text{sim}$ into the triplet form cancels the constants and gives the ranking form exactly.
We verify that below, then show they diverge under **Euclidean** distance — which is what the
SBERT paper actually used for its triplet experiments (§3: *"As metric we use Euclidean distance
and we set ε = 1"*).


In [8]:
import torch
import torch.nn as nn
import torch.nn.functional as F

anchor   = torch.tensor([[1.0, 1.0]])
positive = torch.tensor([[1.2, 0.9]])   # similar to anchor
negative = torch.tensor([[3.0, -1.0]])  # dissimilar

margin  = 0.5
loss_fn = nn.MarginRankingLoss(margin=margin)

sim_pos = F.cosine_similarity(anchor, positive)
sim_neg = F.cosine_similarity(anchor, negative)
target  = torch.ones(1)  # we want sim_pos > sim_neg
# MarginRankingLoss is max(0, -y*(x1 - x2) + margin).
# We pass y = +1 to say "x1 should rank above x2", i.e. sim_pos > sim_neg,
# which expands to max(0, sim_neg - sim_pos + margin).

loss = loss_fn(sim_pos, sim_neg, target)

print("=== (a) Margin Ranking Loss - the SIMILARITY form ===")
print(f"sim(anchor, positive) = {sim_pos.item():.4f}  <- higher is better")
print(f"sim(anchor, negative) = {sim_neg.item():.4f}  <- lower is better")
print(f"Margin Ranking Loss   = {loss.item():.4f}")

manual = max(0.0, sim_neg.item() - sim_pos.item() + margin)
print(f"Manual check: max(0, {sim_neg.item():.4f} - {sim_pos.item():.4f} + {margin}) = {manual:.4f}")
print("PyTorch and manual agree" if abs(loss.item() - manual) < 1e-5 else "Mismatch!")

# --- (b) The SAME triplet, written as Triplet Loss over COSINE DISTANCE ---
print("\n=== (b) Triplet Loss over cosine distance - algebraically identical ===")
d_pos_cos = 1 - sim_pos
d_neg_cos = 1 - sim_neg
triplet_cos = torch.clamp(d_pos_cos - d_neg_cos + margin, min=0.0)
print(f"d(a,p) = 1 - {sim_pos.item():.4f} = {d_pos_cos.item():.4f}")
print(f"d(a,n) = 1 - {sim_neg.item():.4f} = {d_neg_cos.item():.4f}")
print(f"Triplet Loss (cosine) = max(0, {d_pos_cos.item():.4f} - {d_neg_cos.item():.4f} + {margin}) "
      f"= {triplet_cos.item():.4f}")
print(f"Same as Margin Ranking Loss ({loss.item():.4f})? "
      f"{'YES - the (1-x) terms cancel' if abs(triplet_cos.item()-loss.item()) < 1e-5 else 'NO'}")

# --- (c) Triplet Loss over EUCLIDEAN distance - what the SBERT paper used ---
print("\n=== (c) Triplet Loss over Euclidean distance (SBERT paper, eps = 1) ===")
triplet_euc_fn = nn.TripletMarginLoss(margin=1.0, p=2)
loss_euc  = triplet_euc_fn(anchor, positive, negative)
d_pos_euc = (anchor - positive).norm()
d_neg_euc = (anchor - negative).norm()
print(f"d(a,p) = {d_pos_euc.item():.4f}   d(a,n) = {d_neg_euc.item():.4f}   margin = 1.0")
print(f"nn.TripletMarginLoss  = {loss_euc.item():.4f}")
print(f"Manual                = {max(0.0, d_pos_euc.item() - d_neg_euc.item() + 1.0):.4f}")

# Both (b) and (c) report loss 0 - this negative is easy under either metric. But "loss 0"
# hides HOW easy. Compare the pre-clamp value (negative = margin satisfied, with room to spare):
slack_cos = (d_pos_cos - d_neg_cos + margin).item()
slack_euc = (d_pos_euc - d_neg_euc + 1.0).item()
print(f"\n  pre-clamp slack, cosine metric   : {slack_cos:+.4f}")
print(f"  pre-clamp slack, Euclidean metric: {slack_euc:+.4f}")
print("  Same verdict (loss 0), very different margins. The metric is part of the loss:")
print("  it decides which triplets count as 'already solved' and which still produce gradient.")

# --- (d) Hard negative: the gradient only fires when the margin is violated ---
print("\n=== (d) Hard negative case ===")
hard_negative = torch.tensor([[1.05, 0.95]])   # points almost exactly where the anchor points
sim_hard_neg  = F.cosine_similarity(anchor, hard_negative)
loss_hard     = loss_fn(sim_pos, sim_hard_neg, target)
print(f"sim(anchor, hard_negative) = {sim_hard_neg.item():.4f}  <- dangerously high")
print(f"Margin Ranking Loss (hard) = {loss_hard.item():.4f}  <- non-zero: gradient fires")
print("With the easy negative the loss was 0 and the model learned NOTHING from that triplet.")
print("That is exactly why hard/semi-hard negative mining matters (Slide 28).")


=== (a) Margin Ranking Loss - the SIMILARITY form ===
sim(anchor, positive) = 0.9899  <- higher is better
sim(anchor, negative) = 0.4472  <- lower is better
Margin Ranking Loss   = 0.0000
Manual check: max(0, 0.4472 - 0.9899 + 0.5) = 0.0000
PyTorch and manual agree

=== (b) Triplet Loss over cosine distance - algebraically identical ===
d(a,p) = 1 - 0.9899 = 0.0101
d(a,n) = 1 - 0.4472 = 0.5528
Triplet Loss (cosine) = max(0, 0.0101 - 0.5528 + 0.5) = 0.0000
Same as Margin Ranking Loss (0.0000)? YES - the (1-x) terms cancel

=== (c) Triplet Loss over Euclidean distance (SBERT paper, eps = 1) ===
d(a,p) = 0.2236   d(a,n) = 2.8284   margin = 1.0
nn.TripletMarginLoss  = 0.0000
Manual                = 0.0000

  pre-clamp slack, cosine metric   : -0.0427
  pre-clamp slack, Euclidean metric: -1.6048
  Same verdict (loss 0), very different margins. The metric is part of the loss:
  it decides which triplets count as 'already solved' and which still produce gradient.

=== (d) Hard negative case =

## 7. Building a Vector Index with FAISS
*(Slide 41)*

**FAISS (Facebook AI Similarity Search)** is the standard library for fast nearest-neighbour search over large embedding collections.

| Step | Action | Why |
|---|---|---|
| 1 | Encode sentences → float32 matrix `[n, d]` | FAISS needs numpy float32 |
| 2 | `faiss.normalize_L2(vectors)` | Makes inner product = cosine similarity |
| 3 | `IndexFlatIP(d)` | Exact brute-force IP search (no approximation) |
| 4 | `index.add(vectors)` | Stores all vectors |
| 5 | `index.search(query_vec, k)` | Returns `D` (scores), `I` (indices) |

> **Scaling up is not a one-word swap.** `IndexHNSWFlat` is *not* a drop-in replacement for
> `IndexFlatIP`: it takes a graph-connectivity parameter `M`, and it defaults to **L2**, so the
> `D` it returns are squared distances where **lower is better** — the opposite of the cosine
> scores you get here. To keep this notebook's semantics you must ask for inner product explicitly:
> `faiss.IndexHNSWFlat(d, 32, faiss.METRIC_INNER_PRODUCT)`. And per Slide 40, HNSW's speed is
> *empirically* sublinear — it depends on `M`, `efSearch`, and your data distribution, so measure
> Recall@k and latency rather than trusting a complexity class.


In [9]:
import faiss
import numpy as np

# --- A small corpus of 8 sentences ---
corpus = [
    "BERT is a transformer-based language model pre-trained on large text data.",
    "Sentence-BERT produces fixed-size sentence embeddings via siamese networks.",
    "FAISS enables fast nearest-neighbour search over millions of vectors.",
    "Hard negative mining selects negatives close to the anchor during training.",
    "The cat sat on the mat.",
    "Dogs and cats are common household pets.",
    "Deep learning requires large amounts of labelled training data.",
    "Cosine similarity measures the angle between two embedding vectors.",
]

# Step 1 - Encode with a production SBERT model (float32 numpy array)
encode_model = SentenceTransformer('all-MiniLM-L6-v2')
vectors = encode_model.encode(corpus, convert_to_numpy=True).astype(np.float32)
print(f"Step 1 - Encoded: {vectors.shape}  (n={vectors.shape[0]}, dim={vectors.shape[1]})")
print(f"         Norms BEFORE normalize: {np.linalg.norm(vectors, axis=1).round(4)[:4].tolist()} ...")

# Step 2 - L2-normalize so inner product = cosine similarity
faiss.normalize_L2(vectors)
print("Step 2 - L2-normalization")
print(f"         Norms AFTER  normalize: {np.linalg.norm(vectors, axis=1).round(4)[:4].tolist()} ...")

# ...which changed nothing! Why? Because THIS checkpoint already ships a Normalize module,
# so .encode() handed us unit vectors. That is a property of `all-MiniLM-L6-v2`, NOT of
# .encode() in general -- `normalize_embeddings` defaults to False, and models like BGE/E5
# loaded through raw transformers do NOT come back normalized.
# Here is what Step 2 looks like when the model has NOT normalized for you:
raw_vectors = mean_pooling(
    model(**tokenizer(corpus, padding=True, truncation=True, return_tensors="pt")).last_hidden_state,
    tokenizer(corpus, padding=True, truncation=True, return_tensors="pt")['attention_mask']
).detach().numpy().astype(np.float32)
print()
print("  Same Step 2, but on un-normalized raw-BERT vectors:")
print(f"    Norms BEFORE: {np.linalg.norm(raw_vectors, axis=1).round(2)[:4].tolist()} ...  <- wildly different lengths")
faiss.normalize_L2(raw_vectors)
print(f"    Norms AFTER : {np.linalg.norm(raw_vectors, axis=1).round(2)[:4].tolist()} ...  <- all 1.0")
print("  Skip this step on vectors like those and IndexFlatIP ranks by LENGTH, not meaning.")
print("  So: always normalize explicitly. Never assume the model did it for you.")

# Step 3 - Build a flat (exact) inner-product index or dot-product index (IP = Inner Product)
d     = vectors.shape[1]
index = faiss.IndexFlatIP(d)

# Step 4 - Add all vectors to the index
index.add(vectors)
print(f"\nStep 3/4 - Index built: {index.ntotal} vectors stored")

# Step 5 - Search: find top-3 most similar sentences to the query
query_text = "What is a fast library for vector similarity search?"
query_vec  = encode_model.encode([query_text], convert_to_numpy=True).astype(np.float32)
faiss.normalize_L2(query_vec)

D, I = index.search(query_vec, k=3)   # D = cosine scores, I = corpus indices

print(f"\nStep 5 - Query: '{query_text}'\n")
print(f"{'Rank':<5} {'Score':>7}   Document")
print("-" * 70)
for rank, (score, idx) in enumerate(zip(D[0], I[0]), start=1):
    print(f"  {rank}    {score:.4f}   {corpus[idx]}")


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Step 1 - Encoded: (8, 384)  (n=8, dim=384)
         Norms BEFORE normalize: [1.0, 1.0, 1.0, 1.0] ...
Step 2 - L2-normalization
         Norms AFTER  normalize: [1.0, 1.0, 1.0, 1.0] ...

  Same Step 2, but on un-normalized raw-BERT vectors:
    Norms BEFORE: [10.119999885559082, 9.970000267028809, 9.84000015258789, 8.9399995803833] ...  <- wildly different lengths
    Norms AFTER : [1.0, 1.0, 1.0, 1.0] ...  <- all 1.0
  Skip this step on vectors like those and IndexFlatIP ranks by LENGTH, not meaning.
  So: always normalize explicitly. Never assume the model did it for you.

Step 3/4 - Index built: 8 vectors stored

Step 5 - Query: 'What is a fast library for vector similarity search?'

Rank    Score   Document
----------------------------------------------------------------------
  1    0.5753   FAISS enables fast nearest-neighbour search over millions of vectors.
  2    0.5050   Cosine similarity measures the angle between two embedding vectors.
  3    0.2321   BERT is a transformer-b

## Summary

You have now run all **7** demonstration sections, each tied directly to a lecture slide:

| Section | Content | Slide |
|---|---|---|
| 2 | Anisotropy — raw BERT `[CLS]` ranks an unrelated pair *above* a related one | Demo Slide 12 |
| 2b | The cone, measured: raw BERT's pairwise cosine never drops below ≈0.46 over 66 pairs | Slide 19 |
| 3 | Masked mean pooling — and why `[PAD]` outputs are large, not near-zero | Demo Slide 16 |
| 4 | `sentence-transformers` API — 2 lines, production models | Slide 20 |
| 5 | The manual 5-step SBERT *pipeline* + L2 normalize + library verification | Slide 21 |
| 6 | Triplet Loss (cosine and Euclidean) vs. Margin Ranking Loss | Slides 27, 29 |
| 7 | FAISS index — normalize → `IndexFlatIP` → search → top-k | Slide 41 |

**In the Lab**, you will connect all of this into a full semantic search system: encode a real corpus, build a FAISS index, compare SBERT vs. raw BERT search quality, and score the retrieval with Recall@k and MRR. The FAISS index you build will be reused in the **Week 12 RAG lab**.
